In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import statistics

warnings.filterwarnings("ignore")

# Load Data and Inspect
df = pd.read_csv("train.csv")
print(df.info())
print("\nMissing values before cleaning:\n", df.isna().sum())

# Mean Imputation
df['Time_spent_Alone'].fillna(
    df['Time_spent_Alone'].mean(), inplace=True
)

# Mode Imputation
df['Stage_fear'].fillna(
    statistics.mode(df['Stage_fear'].dropna()), inplace=True
)

# Median Imputation
df['Friends_circle_size'].fillna(
    df['Friends_circle_size'].median(), inplace=True
)

# Box Plot Before Outlier Treatment
plt.figure(figsize=(6, 4))
plt.boxplot(df["Friends_circle_size"])
plt.title("Before Outlier Treatment")
plt.show()

# Calculate IQR
def compute_iqr(data, col):
    values = data[col].dropna()
    q1, q3 = np.percentile(values, 25), np.percentile(values, 75)
    iqr = q3 - q1
    lower = q1 - (1.5 * iqr)
    upper = q3 + (1.5 * iqr)
    return q1, q3, lower, upper

# Handle Outliers
def handling_outlier(data, col):
    q1, q3, lower, upper = compute_iqr(data, col)

    low = data[col] < lower
    high = data[col] > upper

    if low.any():
        data.loc[low, col] = np.random.randint(
            int(lower), max(int(q1), int(lower) + 1),
            size=low.sum()
        )

    if high.any():
        data.loc[high, col] = np.random.randint(
            int(q3), max(int(upper), int(q3) + 1),
            size=high.sum()
        )

    return data

df = handling_outlier(df, 'Friends_circle_size')

# Final Check
print("\nMissing values after processing:\n",
      df['Friends_circle_size'].isna().sum())

# Box Plot After Outlier Treatment
plt.figure(figsize=(6, 4))
plt.boxplot(df["Friends_circle_size"])
plt.title("After Outlier Treatment")
plt.show()
